# SPARE: Steering via Pre-trained SAE Representations Demo

This notebook demonstrates SPARE using the unified steering module.

**Paper**: "SPARE: Steering via Pre-trained SAE Representations for Knowledge Conflicts"

**Key idea**: SPARE uses mutual information to select task-relevant SAE features:
$$h' = h + \alpha(-g_\phi(z^-) + g_\phi(z^+))$$

where:
- $z^- = \min\{z_i, z_i^C\}$ (features to remove)
- $z^+ = \max\{z_i^M - z_i, 0\}$ (features to add)

**Datasets**: NQSwap, Macnoise (knowledge conflict) - **NOT IN REGISTRY**
Using sycophancy/ai-risk as proxy datasets.

In [ ]:
# SPARE: Steering via Pre-trained SAE Representations
# Using the unified steering module

import os
import torch
import numpy as np

from Steering import SteeringPipeline

# NOTE: SPARE paper uses NQSwap and Macnoise datasets for knowledge conflict.
# These datasets are NOT in the current data_registry.py.
# TODO: Add to data_registry.py:
#   - "nqswap": {"file": "knowledge_conflict/nqswap.jsonl", "schema": "knowledge_conflict"}
#   - "macnoise": {"file": "knowledge_conflict/macnoise.jsonl", "schema": "knowledge_conflict"}

## 1. Initialize Pipeline

In [ ]:
# Create pipeline with SAE
pipeline = SteeringPipeline(
    model_name="google/gemma-2-2b",
    device="cuda",
    dtype=torch.bfloat16,
)

# Authenticate and load model with SAE
pipeline.authenticate()
pipeline.load_model(use_sae_transformer=True)

# Load SAE for target layer
TARGET_LAYER = 14

pipeline.load_sae(layer=TARGET_LAYER, width="65k")

## 2. Load Dataset

**Note**: SPARE paper uses NQSwap/Macnoise which are NOT in current registry.
Using sycophancy as proxy dataset to demonstrate the method.

In [ ]:
# Load proxy dataset (sycophancy) since NQSwap/Macnoise not in registry
# TODO: Replace with actual knowledge conflict dataset once added to registry:
#   target_data, contrast_data = pipeline.load_data("nqswap", "conflict", n_samples=200)

DATASET_KEY = "sycophancy"
DATASET_VARIANT = "nlp"

target_data, contrast_data = pipeline.load_data(
    dataset_key=DATASET_KEY,
    variant=DATASET_VARIANT,
    n_samples=200,
)

print(f"Loaded {len(target_data)} target samples (proxy for contextual knowledge)")
print(f"Loaded {len(contrast_data)} contrast samples (proxy for parametric knowledge)")
print(f"\nNote: For proper SPARE evaluation, add NQSwap dataset to data_registry.py")

## 3. Extract SPARE Steering Vector

SPARE uses mutual information to select top-k features:
1. Collect SAE activations from demonstrations
2. Compute $I(Z_i; Y)$ for each feature
3. Select features with top-k mutual information

In [ ]:
# Extract steering information
TARGET_LAYER = 14

steering_info = pipeline.extract(
    method="SPARE",
    target_data=target_data,
    contrast_data=contrast_data,
    layer=TARGET_LAYER,
    top_k_proportion=0.01,  # Proportion of MI for feature selection (K in paper)
)

print(f"Extracted SPARE features")
if hasattr(pipeline.extractor, 'selected_features'):
    print(f"Selected features: {len(pipeline.extractor.selected_features)}")

## 4. Create Steered Model and Generate

SPARE steering modifies SAE activations:
$$z'_i = \begin{cases}
z_i - z_i^C + z_i^M & \text{if } i \in \text{selected features} \\
z_i & \text{otherwise}
\end{cases}$$

In [ ]:
# Setup SPARE steered model
pipeline.steering(method="SPARE", layer=TARGET_LAYER)

# Test prompts
TEST_PROMPT = "What is the capital of France? Based on the context: Paris is a small town in Texas."

print(f"Prompt: {TEST_PROMPT}\n")
print("=" * 60)

print("\nBaseline (no steering):")
print(pipeline.generate(TEST_PROMPT, coeff=0.0, max_new_tokens=60, apply_steer=False))

print("\nSteered (coeff=1.0):")
print(pipeline.generate(TEST_PROMPT, coeff=1.0, max_new_tokens=60))

## 5. Coefficient Sweep

In [ ]:
# Test with different steering coefficients
COEFFICIENTS = [-2.0, -1.0, 0.0, 1.0, 2.0]

print(f"Testing with prompt: {TEST_PROMPT[:50]}...\n")
print("=" * 80)

for coeff in COEFFICIENTS:
    if coeff == 0.0:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=40, apply_steer=False)
    else:
        output = pipeline.generate(TEST_PROMPT, coeff=coeff, max_new_tokens=40)
    print(f"\nCoeff = {coeff:+.1f}:")
    print(output[:120] + "..." if len(output) > 120 else output)